# A/C0/C — đánh giá khóa trên val_extra4000_cg · T4×2
Không training, không rerank, không thêm semantic probe. Cùng4000ảnhval ngoài currentdev1000.
Đây là validation đã dùng trong lịch sử, không phải test độc lập mới.
Chạy notebook SAVE_CONCEPT_C_C0_V111_V112_DATASET trước (CPU) để có2checkpoint.
Add Input: dataset weights vừa tạo, pair-source-fullframe-ce(A), prompt-cache,
visual-cache và mscoco-2014(vuthetam). Không cần detector/RGB/UPT cache.
Internet ON, GPU T4×2, Save & Run All. Paths ở cell đầu.
User báo14hGPU; ETA50–100phút nếuI/O tương tự, parentcap2h20, không guarantee.
GPU bật trong CPU phases vẫn có thể tính quota. Không tự tăngcap/đổiảnh khi thiếuinput.
16devsentinels chỉ kiểm parity caption, không thuộc4000ảnhmetrics.


In [ ]:
from pathlib import Path
ARM='EXPANDED_EVAL'
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
WEIGHTS=Path('/kaggle/input/datasets/ducanh2403/concept-mass-c-c0-v111-v112')
CHECKPOINT_PATHS={'A':CHECKPOINT,'C':WEIGHTS/'C.bin','C0':WEIGHTS/'C0.bin'}
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
OUTPUT=Path('/kaggle/working/concept_mass_expanded_eval_4000')
REPORTED_REMAINING_GPU_HOURS=14


In [ ]:
SOURCE_COMMIT='66456737ca25448f8e7e975e45785372040a4fcc'
SOURCE_ASSETS={'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/decoder_relation_memory.py': 'bc5086915bf8f35cf7161311785cf612b25695c5f0dbe03a760d8aca724dfc82', 'captioning/grounded_relations.py': '2c6b15048dce17a9bfa596f02af78866f0b7978f65d172414a3648c9aa49c58f', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'experiments/coco_grounding_feasibility/MAPPING_POLICY_V11.json': 'fd13046512449449375f94ecda44237a33d5a9223813d036fc6d76f4294abfab', 'experiments/concept_mass_grounding/DEV_DIAGNOSTIC_GT.json.gz': '4273bb0572e1d7358b3fb885d1bbb8505b29af5db9356adc3670ff33b21b3f01', 'experiments/concept_mass_grounding/EXPANDED_EVAL_MANIFEST.json': '8f68ab9c45ab9c2d11618e837b8d945f20c0c3148c3b1705ff352cdf244a6fab', 'experiments/concept_mass_grounding/EXPANDED_EVAL_PROTOCOL.md': '4ff68970da4ac543b6d2421bb3f13495f6904d402c14588172ca83bb716c6ea1', 'experiments/concept_mass_grounding/EXPANDED_EVAL_SENTINELS.json': 'f74a8c24f542b859b682316cd37158acf3a0a302d3f0286521c68e7f8030d33d', 'experiments/concept_mass_grounding/TRAIN_PRESENCE.json.gz': 'dd888949a1a8cc363efd760eb46012a7a6f736ac6b167d98be5303393b614117', 'experiments/concept_mass_grounding/concept_data.py': 'e06001f4e1683bb28a6db8a9a85e865e1ebb5c94dcdaa1c5af93901b2d177595', 'experiments/concept_mass_grounding/concept_model.py': 'c5c45d6ba506cd9fec99b406cf19793643a189bb23196bef1c96d6adb84714d0', 'experiments/concept_mass_grounding/diagnostic_concept.py': '45286ed3e07006f331a6091ed67768b5f19fb31adfa4bf90f90099f12c9bb894', 'experiments/concept_mass_grounding/evaluate_concept.py': 'a8a7ad23f58d66921154e3ec763dbc0c16816f6c9e6513b2ba10e6060a36281c', 'experiments/concept_mass_grounding/expanded_data.py': 'e1818410512ad58f4be3429752698396d4c10089efb55401b22b5bcd1b81155e', 'experiments/concept_mass_grounding/expanded_metrics.py': 'bbf4e122cdcfee812539fc0827bc67d3d3ed4dba6d38c5893543c932fbdcda44', 'experiments/concept_mass_grounding/package_eval_checkpoints.py': '0c841acbf4bbb1330c34318aadd1d1ec046e2bceea78c1b487f8fabe3ab21165', 'experiments/concept_mass_grounding/prepare_concept.py': '472ea3708f3897862d2860613c71ff02c80ce9c3565ed9b382b5edebc65aa1a7', 'experiments/concept_mass_grounding/recover_stage.py': 'd3d16ed410f7dc4ce0630a2dc74700642daed495134e010c37e8dc15c41699b4', 'experiments/concept_mass_grounding/run_concept.py': 'b97d06d672c2861015ff0a2e8a835872ec223a03fdb8c301e65b7defe7b6866d', 'experiments/concept_mass_grounding/run_expanded_eval.py': '4ae071e02e03114c5987b09e36c8c601422cf561e4ed67738fc529a92c4ca180', 'experiments/concept_mass_grounding/test_concept_data.py': 'fc68b3992e6018af29123195b8bda527063303ad319a37a940c3138b1be10eee', 'experiments/concept_mass_grounding/test_expanded_eval.py': 'c343bae4425c4e042978d9fb83843980ef55d391f18ed28765eb9dfbbb1fff50', 'experiments/concept_mass_grounding/test_recover_stage.py': 'd7892615af99c51dba31f57beab12ffc704f65b874198d706198e74b2688df8e', 'experiments/relation_frame_pretrain/frame_data.py': '87c58a50142f68b1721323f7c53fa08e256bb5bc8aee3d7625262161ae1692de', 'experiments/semantic_grounding_method/data/gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67', 'experiments/shared_pair_readout/metric_preflight.py': '6e29f54e57b0b2e7f6f62de57017061071444d254b09305411716713b86f83f4', 'experiments/shared_pair_readout/readout_data.py': '3bdc5e38c942cfeb13f8a83e409e367d6cba1b2fb5da6149c2d400b0c51b889c'}
import hashlib,json,os,signal,subprocess,sys,tempfile,threading,time,traceback,zipfile
os.environ.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',
    PYTHONUNBUFFERED='1',PYTHONHASHSEED='42',TOKENIZERS_PARALLELISM='false',HDF5_USE_FILE_LOCKING='FALSE')
PARENT_DEADLINE=time.monotonic()+8400;PHASE_DEADLINE=min(PARENT_DEADLINE,time.monotonic()+900)
REPO=Path(tempfile.mkdtemp(prefix='concept-mass-'+ARM+'-'))/'repo'
LOG=OUTPUT.with_name(OUTPUT.name+'_notebook.log');REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
for p in (OUTPUT,LOG,REVIEW):
    if p.exists():raise FileExistsError('Giữ output/log cũ; dùng phiên mới: '+str(p))
for p in (COCO_JSON,LEGACY_PROMPT,*CHECKPOINT_PATHS.values()):
    if not p.is_file():raise FileNotFoundError('Add Input chứa '+str(p))
if not VISUAL_CACHE.is_dir():raise FileNotFoundError(VISUAL_CACHE)
def run(args,cwd=None,timeout=900):
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic(),PHASE_DEADLINE-time.monotonic())
    if timeout<=0:raise TimeoutError("Parent/phase budget exhausted")
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf8') as log:
        log.write('Running: '+' '.join(args)+'\n');log.flush()
        process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1,start_new_session=True)
        def drain():
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        reader=threading.Thread(target=drain,daemon=True);reader.start();start=time.monotonic()
        try:
            while process.poll() is None:
                if time.monotonic()-start>timeout:raise TimeoutError('Phase cap; incomplete')
                reader.join(timeout=20)
                if process.poll() is None:print('Worker active;',round(time.monotonic()-start),'s elapsed; see stage/step logs.',flush=True)
            reader.join()
            if process.returncode:raise subprocess.CalledProcessError(process.returncode,args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            reader.join(timeout=5)
print('Bootstrap started; input paths found.',flush=True)


In [ ]:
try:
    run(['git','init',REPO]);run(['git','remote','add','origin','https://github.com/Supzxjee/Image_Captioning.git'],cwd=REPO)
    run(['git','fetch','--depth','1','origin',SOURCE_COMMIT],cwd=REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO)
    if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=SOURCE_COMMIT:raise ValueError('Wrong source commit')
    for name,h in SOURCE_ASSETS.items():
        if hashlib.sha256((REPO/name).read_bytes()).hexdigest()!=h:raise ValueError('Changed source '+name)
    os.environ['PYTHONPATH']=str(REPO);HERE=REPO/'experiments/concept_mass_grounding'
    run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocoevalcap==1.2','h5py','numpy','Pillow','nltk==3.9.2','future'])
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
    run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
    run(['java','-version']);CHAIR=REPO.parent/'chair'
    run(['git','init',CHAIR]);run(['git','remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],cwd=CHAIR)
    run(['git','fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],cwd=CHAIR);run(['git','checkout','--detach','FETCH_HEAD'],cwd=CHAIR)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')
    sys.path.insert(0,str(REPO));sys.path.insert(0,str(HERE))
    import importlib.util
    policy_spec=importlib.util.spec_from_file_location('concept_mass_policy_'+SOURCE_COMMIT,HERE/'expanded_data.py')
    policy_module=importlib.util.module_from_spec(policy_spec);policy_spec.loader.exec_module(policy_module)
    POLICY=policy_module.EVAL_POLICY
    CONFIG=REPO.parent/'config.json'
    cfg=dict(checkpoints={k:str(v) for k,v in CHECKPOINT_PATHS.items()},dataset_json=str(COCO_JSON),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),metadata=str(REPO/'experiments/semantic_grounding_method/data/gain_metadata.json.gz'),chair=str(CHAIR),output=str(OUTPUT),policy=POLICY,source_commit=SOURCE_COMMIT,source_assets=SOURCE_ASSETS,reported_remaining_gpu_hours=REPORTED_REMAINING_GPU_HOURS)
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf8')

except BaseException:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.writestr('execution_status.json',json.dumps(dict(success=False,arm=ARM,stage='bootstrap',source_commit=SOURCE_COMMIT)))
        z.writestr('bootstrap_error.txt',traceback.format_exc())
    print('BOOTSTRAP FAILED; SEND:',REVIEW,flush=True)
    raise


In [ ]:
success=False
try:
    PHASE_DEADLINE=min(PARENT_DEADLINE,time.monotonic()+3720)
    run([sys.executable,'-u',HERE/'test_expanded_eval.py'],cwd=REPO,timeout=120)
    run([sys.executable,'-u',REPO/'experiments/shared_pair_readout/metric_preflight.py'],cwd=REPO,timeout=180)
    run([sys.executable,'-u',HERE/'run_expanded_eval.py','prepare','--config',CONFIG],cwd=REPO,timeout=3720)
    PHASE_DEADLINE=min(PARENT_DEADLINE,time.monotonic()+1800)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',HERE/'run_expanded_eval.py','worker','--config',CONFIG],cwd=REPO,timeout=1800)
    PHASE_DEADLINE=min(PARENT_DEADLINE,time.monotonic()+1800)
    run([sys.executable,'-u',HERE/'run_expanded_eval.py','evaluate','--config',CONFIG],cwd=REPO,timeout=1800)
    result=json.loads((OUTPUT/'summary.json').read_text())
    if result['status']!='EXPANDED_EVALUATION_COMPLETE' or result['count']!=4000:raise RuntimeError('Incomplete evaluation')
    success=True
finally:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.write(CONFIG,'notebook_config.json')
        for name in SOURCE_ASSETS:
            if name.endswith('.py'):z.write(REPO/name,'source/'+name)
        if OUTPUT.is_dir():
            for p in OUTPUT.iterdir():
                if p.suffix in ('.json','.jsonl','.log'):z.write(p,p.name)
        z.writestr('execution_status.json',json.dumps(dict(success=success,source_commit=SOURCE_COMMIT,assistant_launched_job=False,training=False)))
    print('SEND REVIEW ZIP:',REVIEW,flush=True)
    if not success:print('INCOMPLETE: send log/ZIP; do not score a subset as complete.',flush=True)
if success:print('DONE: locked evaluation only; do not tune on this cohort.',flush=True)


Gửi concept_mass_expanded_eval_4000_review.zip. Không có checkpoint mới vì khôngtraining.
Báo mọi contrasts/gates đã khóa; nếu không hỗ trợ, dừngclaim đợt này, khôngtuning trên4000ảnh.
Notebook chưa được assistant chạy GPU/API/model; actual parity/metrics chờ lượt của bạn.
